# 01 · Bronze ingest
Lands every extract batch **as-is** (all strings) into Delta tables with lineage columns. Idempotent: a batch log skips loaded batches and a half-loaded batch is deleted and reloaded. `mergeSchema` absorbs new source columns (schema drift).

In [ ]:
from pyspark.sql import functions as F

TABLES = ["customer", "vendor", "item", "location", "sales_header", "sales_line",
          "sales_invoice_header", "sales_invoice_line", "sales_cr_memo_header",
          "sales_cr_memo_line", "item_ledger_entry", "sales_header_keys"]

ROOT = "Files/bc_extract"   # folder that contains full_2026-09-30, incr_2026-10-01, ...

spark.sql("""CREATE TABLE IF NOT EXISTS etl_batch_log (batch_id STRING, loaded_at TIMESTAMP) USING DELTA""")

done = {r.batch_id for r in spark.table("etl_batch_log").collect()}
batches = sorted(f.name.strip("/") for f in mssparkutils.fs.ls(ROOT) if f.isDir)

print("Batches found:  ", batches)
print("Already loaded: ", sorted(done))
print("To load now:    ", [b for b in batches if b not in done])

In [ ]:
for batch in batches:
    if batch in done:
        print(f"skip {batch} (already loaded)")
        continue

    for t in TABLES:
        path = f"{ROOT}/{batch}/{t}.csv"
        if not mssparkutils.fs.exists(path):
            continue

        df = (spark.read
                .option("header", True)
                .option("inferSchema", False)            # Bronze = everything as text
                .csv(path)
                .withColumn("_source_file", F.col("_metadata.file_path"))
                .withColumn("_batch_id", F.lit(batch))
                .withColumn("_ingested_at", F.current_timestamp()))

        target = f"bronze_{t}"
        if spark.catalog.tableExists(target):            # safe rerun of a half-loaded batch
            spark.sql(f"DELETE FROM {target} WHERE _batch_id = '{batch}'")

        (df.write.format("delta").mode("append")
           .option("mergeSchema", "true")                # schema drift
           .saveAsTable(target))
        print(f"{batch:18} {target:30} {df.count():>6} rows")

    spark.sql(f"INSERT INTO etl_batch_log VALUES ('{batch}', current_timestamp())")

In [ ]:
expected = {
    "bronze_customer": 436, "bronze_vendor": 16, "bronze_item": 186, "bronze_location": 9,
    "bronze_sales_header": 86, "bronze_sales_line": 176, "bronze_sales_invoice_header": 3461,
    "bronze_sales_invoice_line": 6538, "bronze_sales_cr_memo_header": 110,
    "bronze_sales_cr_memo_line": 110, "bronze_item_ledger_entry": 11176,
    "bronze_sales_header_keys": 122,
}
for t, exp in expected.items():
    n = spark.table(t).count()
    print(f"{t:32} {n:>6}  {'OK' if n == exp else f'EXPECTED {exp}'}")

In [ ]:
%%sql
-- schema drift: Delivery_Method only exists in the last batch
SELECT _batch_id, COUNT(*) AS rows, COUNT(Delivery_Method) AS rows_with_delivery_method
FROM bronze_sales_header
GROUP BY _batch_id
ORDER BY _batch_id